In [ ]:
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# Helpers
def tokenize(text: str) -> list[str]:
    text = re.sub(r"[^a-z0-9]+", " ", text.lower())
    tokens = text.split()
    return [tok for tok in tokens if tok]


def top_words(tokens: list[str], n: int = 20) -> pd.DataFrame:
    counts = Counter(tokens)
    common = counts.most_common(n)
    return pd.DataFrame(common, columns=["word", "count"])


In [ ]:
# Question 1
df_qa = pd.read_json("qa_Musical_Instruments.json", lines=True)

question_tokens = []
answer_tokens = []

for _, row in df_qa.iterrows():
    question_tokens.extend(tokenize(str(row.get("question", ""))))
    answer_tokens.extend(tokenize(str(row.get("answer", ""))))

q1_top_questions = top_words(question_tokens, 20)
q1_top_answers = top_words(answer_tokens, 20)

q1_top_questions


In [ ]:
question_texts = df_qa["question"].fillna("").astype(str)

question_word_counts = question_texts.apply(lambda t: len(tokenize(t)))
question_char_counts = question_texts.apply(len)

avg_words = question_word_counts.mean()
avg_chars = question_char_counts.mean()

print(f"Average question length (words): {avg_words:.2f}")
print(f"Average question length (characters): {avg_chars:.2f}")


In [ ]:
answer_texts = df_qa["answer"].fillna("").astype(str)
answer_word_counts = answer_texts.apply(lambda t: len(tokenize(t)))

bucketed = answer_word_counts.apply(lambda n: 30 if n > 30 else n)
counts = bucketed.value_counts().to_dict()

x_labels = [str(i) for i in range(1, 31)] + ["30+"]
y_values = [counts.get(i, 0) for i in range(1, 31)] + [counts.get(30, 0)]

plt.figure(figsize=(10, 4))
plt.bar(range(len(x_labels)), y_values, color="#4c78a8")
plt.xticks(range(len(x_labels)), x_labels, rotation=90)
plt.xlabel("Answer length (words)")
plt.ylabel("Number of answers")
plt.title("Histogram of Answer Lengths (Words)")
plt.tight_layout()
plt.show()


In [ ]:
#Question 2
df_spam = pd.read_csv("spam.csv", encoding="latin-1")

if "v1" in df_spam.columns and "v2" in df_spam.columns:
    label_col, text_col = "v1", "v2"
else:
    label_col, text_col = df_spam.columns[:2]

df_spam = df_spam[[label_col, text_col]].rename(columns={label_col: "label", text_col: "text"})
df_spam["label"] = df_spam["label"].astype(str).str.lower()

spam_texts = df_spam[df_spam["label"] == "spam"]["text"].fillna("").astype(str)
ham_texts = df_spam[df_spam["label"] == "ham"]["text"].fillna("").astype(str)

spam_tokens = []
ham_tokens = []

for t in spam_texts:
    spam_tokens.extend(tokenize(t))
for t in ham_texts:
    ham_tokens.extend(tokenize(t))

q2_top_spam = top_words(spam_tokens, 20)
q2_top_ham = top_words(ham_tokens, 20)

q2_top_spam


In [ ]:
spam_counts = Counter(spam_tokens)
ham_counts = Counter(ham_tokens)

vocab = sorted(set(spam_counts) | set(ham_counts))
V = len(vocab)

eps = 1e-12
spam_total = sum(spam_counts.values())
ham_total = sum(ham_counts.values())

P = {}
Q = {}
for w in vocab:
    P[w] = (spam_counts[w] + eps) / (spam_total + eps * V)
    Q[w] = (ham_counts[w] + eps) / (ham_total + eps * V)

kl_spam_ham = sum(P[w] * np.log(P[w] / Q[w]) for w in vocab)
kl_ham_spam = sum(Q[w] * np.log(Q[w] / P[w]) for w in vocab)

print(f"KL(spam || ham): {kl_spam_ham:.6f}")
print(f"KL(ham || spam): {kl_ham_spam:.6f}")

spam_unique_scores = [(w, P[w] * np.log(P[w] / Q[w])) for w in vocab]
ham_unique_scores = [(w, Q[w] * np.log(Q[w] / P[w])) for w in vocab]

spam_unique_scores.sort(key=lambda x: x[1], reverse=True)
ham_unique_scores.sort(key=lambda x: x[1], reverse=True)

unique_spam_df = pd.DataFrame(spam_unique_scores[:20], columns=["word", "score"])
unique_ham_df = pd.DataFrame(ham_unique_scores[:20], columns=["word", "score"])

unique_spam_df


In [ ]:
all_texts = df_spam["text"].fillna("").astype(str)
message_sets = [set(tokenize(t)) for t in all_texts]


def cooccurrence_top_words(message_sets, target_words, top_n=5):
    results = {}
    for target in target_words:
        counts = Counter()
        for tokens in message_sets:
            if target in tokens:
                for w in tokens:
                    if w != target:
                        counts[w] += 1
        results[target] = counts.most_common(top_n)
    return results


spam_targets = unique_spam_df["word"].tolist()
ham_targets = unique_ham_df["word"].tolist()

spam_cooccurrence = cooccurrence_top_words(message_sets, spam_targets, top_n=5)
ham_cooccurrence = cooccurrence_top_words(message_sets, ham_targets, top_n=5)

for target, pairs in spam_cooccurrence.items():
    print(f"{target} -> {pairs}")

for target, pairs in ham_cooccurrence.items():
    print(f"{target} -> {pairs}")
